In [ ]:
from google.colab import files
files.upload()    # choose iot23_step1c_rigorous_eval.py

Saving iot23_step1c_rigorous_eval.py to iot23_step1c_rigorous_eval.py


{'iot23_step1c_rigorous_eval.py': b'"""\nSmart Building Digital Forensic Readiness Framework\nStep 1c (IoT-23): Rigorous cross-capture evaluation (nested leave-one-capture-out)\n\nWHY THIS SCRIPT EXISTS\n  Earlier scripts had weaknesses a reviewer would flag:\n    - one random split of the captures (results swung from 0% to 99% recall)\n    - the epoch / threshold were chosen on the same data that was reported\n    - only one model, no baselines, one seed, no uncertainty\n    - "accuracy" and pooled scores hide how badly some captures are handled\n\n  This script fixes those:\n    1. NESTED leave-one-capture-out. Each capture is the test set once. Every\n       choice (early stopping epoch, alert threshold) is made ONLY from\n       validation captures taken from the training side, never from the test\n       capture.\n    2. The alert threshold is an operating point: the score that gives a fixed\n       false-alarm budget (FPR_BUDGET) on the validation captures\' benign flows.\n      

In [ ]:
%run iot23_step1c_rigorous_eval.py


Mounted at /content/drive
Using device: cpu
Models: ['mlp', 'logreg', 'rf', 'hgb'] | seeds: [0, 1, 2] | results go to: /content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs/rigorous_eval

Loading cached sample from /content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs/iot23_sample_cap40000_chunksNone_scNone.pkl
Collapsed 886,853 sampled flows into 153,180 distinct patterns (weights = number of sampled flows per pattern).
Data: 153,180 patterns | benign 84,981 | malicious 68,199 | 23 captures
Test captures (each held out once): 10 -> CTU-IoT-Malware-Capture-1-1, CTU-IoT-Malware-Capture-17-1, CTU-IoT-Malware-Capture-3-1, CTU-IoT-Malware-Capture-34-1, CTU-IoT-Malware-Capture-35-1, CTU-IoT-Malware-Capture-48-1, CTU-IoT-Malware-Capture-49-1, CTU-IoT-Malware-Capture-52-1, CTU-IoT-Malware-Capture-7-1, CTU-IoT-Malware-Capture-8-1

Running nested leave-one-capture-out (every choice is made without the test capture)...

[1/10] CTU-IoT-Malware-Capture-1-1 s

In [ ]:
from google.colab import files
files.upload()    # choose iot23_step1d_improvement_experiments.py

Saving iot23_step1d_improvement_experiments.py to iot23_step1d_improvement_experiments.py


{'iot23_step1d_improvement_experiments.py': b'"""\nSmart Building Digital Forensic Readiness Framework\nStep 1d (IoT-23): Experiments to reduce cross-capture over-fitting and threshold instability\n\nBACKGROUND\n  Step 1c showed (nested leave-one-capture-out, 10 test captures):\n    - training and validation scores differ from epoch 1: the gap comes from the\n      shift BETWEEN captures, not from a model that is too big or trained too long\n    - ranking quality (PR-AUC) is decent, but recall at the chosen alert threshold\n      swings between 0 and 1 from seed to seed (the threshold is unstable)\n    - a simple logistic regression is as good as the MLP\n\nWHAT THIS SCRIPT TRIES (every idea is compared with the baseline under the\nSAME protocol, the SAME inner validation captures and the SAME seeds):\n    capture_balance : every training capture counts equally in the loss, so the\n                      two or three huge captures do not dominate training\n    ratio features  : replaces

In [ ]:
%run iot23_step1d_improvement_experiments.py

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cpu
Models: ['mlp', 'logreg'] | experiments: ['E0_baseline', 'E1_capture_bal', 'E2_ratio_feats', 'E3_ensemble', 'E4_all'] | seeds: [0, 1, 2]
Results go to: /content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs/improvement_experiments

Loading cached sample from /content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs/iot23_sample_cap40000_chunksNone_scNone.pkl
Collapsed 886,853 sampled flows into 153,180 distinct patterns.
Data: 153,180 patterns | benign 84,981 | malicious 68,199 | 23 captures
Test captures (each held out once): 10 -> CTU-IoT-Malware-Capture-1-1, CTU-IoT-Malware-Capture-17-1, CTU-IoT-Malware-Capture-3-1, CTU-IoT-Malware-Capture-34-1, CTU-IoT-Malware-Capture-35-1, CTU-IoT-Malware-Capture-48-1, CTU-IoT-Malware-Capture-49-1, CTU-IoT-Malware-Capture-52-1, CTU-IoT-Malware-Capture-7-1, CTU-IoT-Malw

In [1]:
from google.colab import files
files.upload()    # choose step3_evidence_integrity.py

Saving step3_evidence_integrity.py to step3_evidence_integrity.py


{'step3_evidence_integrity.py': b'"""\nSmart Building Digital Forensic Readiness Framework\nStep 3: Evidence integrity (tamper-proofing)\n\nWHERE THIS FITS\n  Step 1 = detect attacks (ML)          -> iot23_model.pt\n  Step 2 = log every detection          -> detection_log.jsonl\n  Step 3 = PROTECT that log (this file) -> proves it was not altered\n\nWHY STEP 3 EXISTS EVEN THOUGH STEP 2 ALREADY CHAINS HASHES\n  Step 2 gives every log entry a SHA-256 hash that includes the previous\n  entry\'s hash. That catches a careless edit. But it has two blind spots:\n    1. Someone who edits an old entry can simply recompute every hash after\n       it. The chain then looks perfectly valid (anyone can compute SHA-256).\n    2. Someone who deletes the LAST entries leaves a shorter chain that is\n       still perfectly valid.\n  Step 3 closes both gaps with "seals": a seal records how many entries\n  the log had and the last hash at that moment, and is signed with a SECRET\n  KEY (HMAC-SHA256). With

In [2]:
%run step3_evidence_integrity.py demo

Mounted at /content/drive
Action: demo

DEMO - 20 made-up detections in a temporary folder; none of your real files are touched.

Attack                                     Chain alone   Chain + signed seal   Caught by
----------------------------------------------------------------------------------------------------
A. Nothing touched                         ok            ok                    -
B. One field edited                        caught        caught                entry_modified
C. One entry deleted from the middle       caught        caught                chain_broken
D. Last 5 entries deleted                  MISSED        caught                log_truncated
E. Entry edited + ALL hashes recomputed    MISSED        caught                history_rewritten
F. E + seal edited to match                MISSED        caught                seal_forged
G. E + seals file deleted                  MISSED        MISSED                -

What this shows:
 - B and C: a plain hash chain is

In [3]:
import os
SAVE_DIR = "/content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs"
print(os.path.exists(f"{SAVE_DIR}/detection_log.jsonl"))

True


In [4]:
%run step3_evidence_integrity.py verify

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Action: verify

INTEGRITY CHECK
Log file : /content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs/detection_log.jsonl
Entries  : 2,711
Head hash: c2ae8bd08b5ae3f927f0c1ea6875843250a54a4363fe07158ca8ed094cbb0817
Seals    : 0 found, 0 valid
Custody  : 0 records, chain OK
  - [entry_modified] Entry 1356 (line 1356) was changed after it was written: its content no longer matches its hash.

RESULT: FAIL ❌  the evidence was altered

Done.


In [5]:
import shutil
shutil.copy(f"{SAVE_DIR}/detection_log_backup.jsonl", f"{SAVE_DIR}/detection_log.jsonl")

'/content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs/detection_log.jsonl'

In [6]:
%run step3_evidence_integrity.py all

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Action: all

⚠️  A new integrity key was created:
   /content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs/integrity/integrity.key
   Whoever has this key can make valid-looking seals. Copy it somewhere SAFE and SEPARATE now;
   if you lose it, existing seals can no longer be checked.

INTEGRITY CHECK
Log file : /content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs/detection_log.jsonl
Entries  : 2,711
Head hash: c2ae8bd08b5ae3f927f0c1ea6875843250a54a4363fe07158ca8ed094cbb0817
Seals    : 0 found, 0 valid
Custody  : 1 records, chain OK

RESULT: PASS (chain only) ⚠️  hash chain is intact, but the log has NEVER BEEN SEALED - a full rewrite or deleted tail would not be noticed. Run the seal action.

✅ Sealed 2,711 entries (seal b096e660...).
   Head hash: c2ae8bd08b5ae3f927f0c1ea6875843250a54a4363fe07158ca8ed094cbb0817
   Se

In [7]:
from google.colab import files
files.download(f"{SAVE_DIR}/integrity/integrity.key")
files.download(f"{SAVE_DIR}/integrity/integrity_seals.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
import shutil
LOG = f"{SAVE_DIR}/detection_log.jsonl"
shutil.copy(LOG, "/content/log_good.jsonl")                # backup
lines = open(LOG).read().splitlines()
open(LOG, "w").write("\n".join(lines[:-5]) + "\n")         # delete the last 5 entries
%run step3_evidence_integrity.py verify                    # expect FAIL: log_truncated
shutil.copy("/content/log_good.jsonl", LOG)                # restore
%run step3_evidence_integrity.py verify                    # expect PASS

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Action: verify

INTEGRITY CHECK
Log file : /content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs/detection_log.jsonl
Entries  : 2,706
Head hash: a049b902a9478bc234a386a2f28e90ac075a479fdb64253ec57d70615593ea3d
Seals    : 1 found, 0 valid
Custody  : 4 records, chain OK
  - [log_truncated] The log has 2,706 entries but seal 1 certified 2,711: entries were deleted from the end.

RESULT: FAIL ❌  the evidence was altered

Done.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Action: verify

INTEGRITY CHECK
Log file : /content/drive/MyDrive/PBL-Smart Building Digital Forensics/iot23_outputs/detection_log.jsonl
Entries  : 2,711
Head hash: c2ae8bd08b5ae3f927f0c1ea6875843250a54a4363fe07158ca8ed094cbb0817
Seals    : 1 found, 1 valid | latest valid seal covers 2,711 entr